# ☔ Umbrella Prediction using Scikit-Learn (with CoreML Export)

## Problem Description

This notebook revisits the umbrella prediction problem from
[`02-logistic-regression-umbrella.ipynb`](02-logistic-regression-umbrella.ipynb), this time using
scikit-learn instead of a from-scratch implementation. The goal is to export the trained model to
CoreML and use it in a small iOS app.

## Dataset

Same two features (Humidity %, Temperature °C) and target (umbrella needed) as the previous notebook.

## Objective

- Train a logistic regression model with scikit-learn
- Compare the learned parameters against the from-scratch implementation
- Export the trained model to CoreML (`.mlmodel`)
- Use the exported model in a minimal iOS app

In [10]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [11]:
# Sigmoid function
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

In [ ]:
# Generate data
np.random.seed(42) # for reproducibility

number_of_samples = 200
humidity = np.random.uniform(60, 90, number_of_samples)       # humidity in %
temperature = np.random.uniform(-2, 30, number_of_samples)    # temperature °C

bias = -(0.15 * 75 - 0.1 * 14)  # = -9.85
z = 0.15 * humidity - 0.1 * temperature + bias
probability = sigmoid(z)

X_train = np.column_stack((humidity, temperature))
y_train = (np.random.uniform(0, 1, number_of_samples) < probability).astype(int)

In [ ]:
# Logistic Regression

# Bundle preprocessing and classification into a single pipeline, so the 
# exported CoreML model handles normalisation internally — the iOS app 
# can pass raw humidity/temperature values directly.
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression())
])
pipeline.fit(X_train, y_train)

print(f"w (coef_): {pipeline.named_steps['classifier'].coef_}")
print(f"b (intercept_): {pipeline.named_steps['classifier'].intercept_}")

accuracy = pipeline.score(X_train, y_train)
print(f"Training accuracy: {accuracy:.2%}")

w (coef_): [[ 1.514474  -0.7316404]]
b (intercept_): [-0.30397283]
Training accuracy: 79.00%


## Comparison

Scikit-learn's parameters (`w ≈ [1.51, -0.73]`, `b ≈ -0.30`) closely match the
from-scratch version (`w ≈ [1.60, -0.78]`, `b ≈ -0.31`) — slightly smaller due to
scikit-learn's default L2 regularization. Training accuracy: 79%, consistent with
the noisy labels used to generate the data.